# Optimización de Membresías con Algoritmo Genético

**Cosme Botito — Bot de Moderación Automática para Discord**  
*Introducción a la Inteligencia Artificial — UTN*

---

Este notebook explica cómo y por qué se usa un **algoritmo genético** para optimizar los parámetros del sistema de lógica difusa que determina el nivel de toxicidad de un mensaje.


## 1. El problema: demasiados parámetros para ajustar a mano

El sistema difuso de Cosme Botito tiene cuatro variables de entrada:

| Variable | Fuente | Qué mide |
|---|---|---|
| `lista_negra` | Regex propio | Densidad de palabras ofensivas rioplatenses |
| `CONICET` | Pérez et al. (UBA/CONICET) | Hate speech colectivo en rioplatense |
| `detoxify` | Detoxify multilingual | Toxicidad general e insultos interpersonales |
| `historial_usuario` | SQLite | Infracciones previas del usuario |

Y una variable de **salida** con cuatro niveles:

```
baja  →  media  →  alta  →  extrema
```

Cada nivel de la salida es un **trapezoide** definido por 4 parámetros `(a, b, c, d)`:

```
     1 |      /‾‾‾‾‾‾\
       |     /         \
     0 |____/           \____
          a   b       c   d
```

Con 4 niveles × 4 parámetros = **16 números** que definir.

El problema es que cambiar cualquiera de esos 16 números afecta el comportamiento de todo el sistema. Ajustarlos a mano es prácticamente imposible porque:

- El espacio de búsqueda es continuo y de 16 dimensiones
- Los efectos son no lineales e interdependientes
- Cada evaluación requiere correr el sistema sobre 148 mensajes etiquetados

Necesitamos un algoritmo que explore ese espacio automáticamente. La solución: **algoritmo genético**.


## 2. ¿Qué es un algoritmo genético?

Un algoritmo genético (AG) es una técnica de optimización inspirada en la **evolución biológica**. La idea central es simple:

> Si tenemos una población de soluciones candidatas y les aplicamos selección, cruce y mutación iterativamente, las soluciones mejores tienden a sobrevivir y reproducirse, y la población converge hacia mejores soluciones.

Los conceptos clave mapeados a nuestro problema:

| Concepto biológico | En nuestro problema |
|---|---|
| **Individuo** | Un conjunto de 16 parámetros para los trapezoides de toxicidad |
| **Gen** | Uno de los 16 parámetros (un número entre 0 y 1) |
| **Cromosoma** | El vector completo de 16 parámetros |
| **Población** | 60 individuos (configuraciones distintas) |
| **Fitness** | Accuracy del sistema difuso sobre el dataset de prueba |
| **Selección** | Elegir los individuos con mayor accuracy para reproducirse |
| **Cruce** | Combinar dos configuraciones para generar configuraciones nuevas |
| **Mutación** | Perturbar aleatoriamente algunos parámetros |
| **Generación** | Una iteración del proceso completo |


## 3. Representación del individuo

Cada individuo es un vector de 16 números que define los 4 trapezoides de la variable de salida `toxicidad`:

```python
individuo = [
    # baja:    a,    b,    c,    d
              0.00, 0.00, 0.15, 0.28,
    # media:   a,    b,    c,    d
              0.18, 0.28, 0.42, 0.55,
    # alta:    a,    b,    c,    d
              0.42, 0.55, 0.68, 0.78,
    # extrema: a,    b,    c,    d
              0.65, 0.75, 1.00, 1.00,
]
```

Para que un individuo sea **válido**, los parámetros deben cumplir:
- Dentro de cada trapezoide: `a ≤ b ≤ c ≤ d`
- Entre categorías: los picos deben estar ordenados `baja_b < media_b < alta_b < extrema_b`

Esto garantiza que las categorías no se inviertan (no tendría sentido que "extrema" esté a la izquierda de "baja").


## 4. Visualización de un individuo


In [ ]:
import numpy as np
import skfuzzy as fuzz
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Configuración actual (baseline)
BASELINE = [
    0.00, 0.00, 0.15, 0.28,   # baja
    0.18, 0.28, 0.42, 0.55,   # media
    0.42, 0.55, 0.68, 0.78,   # alta
    0.65, 0.75, 1.00, 1.00,   # extrema
]

universe = np.arange(0, 1.01, 0.01)
nombres  = ['baja', 'media', 'alta', 'extrema']
colores  = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c']

fig, ax = plt.subplots(figsize=(10, 4))

for i, (nombre, color) in enumerate(zip(nombres, colores)):
    params = BASELINE[i*4:(i+1)*4]
    y = fuzz.trapmf(universe, params)
    ax.plot(universe, y, color=color, linewidth=2, label=nombre)
    ax.fill_between(universe, y, alpha=0.15, color=color)

ax.set_title('Funciones de membresía de toxicidad (configuración baseline)', fontsize=13)
ax.set_xlabel('Score de toxicidad [0-1]')
ax.set_ylabel('Grado de pertenencia')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.1)

# Marcar los umbrales de acción
for x, label in [(0.28, 'ignorar'), (0.55, 'alertar'), (0.78, 'timeout')]:
    ax.axvline(x, color='gray', linestyle='--', alpha=0.5)
    ax.text(x + 0.01, 0.95, label, fontsize=8, color='gray')

plt.tight_layout()
plt.show()
print("Parámetros baseline:", BASELINE)


## 5. Función de fitness

La función de fitness responde a una sola pregunta:

> **¿Qué tan bien clasifica este individuo los 148 mensajes del dataset?**

```
fitness(individuo) = mensajes_correctamente_clasificados / 148
```

Para cada individuo:
1. Se aplican sus 16 parámetros a los trapezoides de toxicidad
2. Se corre el sistema difuso sobre cada mensaje del dataset
3. Se convierte el score numérico a una categoría (baja/media/alta/extrema)
4. Se compara con la categoría esperada (etiquetada manualmente)
5. Se cuenta el porcentaje de aciertos

Un individuo con fitness = 0.70 clasifica correctamente 104 de 148 mensajes.


## 6. Operadores genéticos

### 6.1 Selección por torneo

Para elegir qué individuos se reproducen, se usa **selección por torneo**:

1. Se eligen 4 individuos al azar de la población
2. El que tiene mayor fitness "gana" y es seleccionado para reproducirse

Esto mantiene diversidad (los peores tienen alguna chance) pero favorece a los mejores.

### 6.2 Cruce en un punto

Dados dos padres, se elige un punto de corte aleatorio y se intercambian las "colas":

```
Padre 1: [0.00, 0.00, 0.15, 0.28 | 0.18, 0.28, 0.42, 0.55, 0.42, 0.55, 0.68, 0.78, 0.65, 0.75, 1.00, 1.00]
Padre 2: [0.00, 0.00, 0.20, 0.35 | 0.22, 0.35, 0.50, 0.62, 0.50, 0.62, 0.75, 0.85, 0.72, 0.82, 1.00, 1.00]
                                  ↑ punto de cruce (posición 4)

Hijo 1:  [0.00, 0.00, 0.15, 0.28 | 0.22, 0.35, 0.50, 0.62, 0.50, 0.62, 0.75, 0.85, 0.72, 0.82, 1.00, 1.00]
Hijo 2:  [0.00, 0.00, 0.20, 0.35 | 0.18, 0.28, 0.42, 0.55, 0.42, 0.55, 0.68, 0.78, 0.65, 0.75, 1.00, 1.00]
```

### 6.3 Mutación gaussiana

Con probabilidad `p = 0.15` por gen, se le suma un ruido gaussiano pequeño:

```python
nuevo_valor = valor_actual + random.gauss(0, 0.05)
```

Esto permite explorar el espacio de búsqueda alrededor de los mejores individuos.

### 6.4 Elitismo

Los 5 mejores individuos de cada generación pasan directamente a la siguiente sin modificarse. Esto garantiza que la solución nunca empeora.


## 7. El algoritmo completo

```
1. INICIALIZACIÓN
   - Crear población de 60 individuos aleatorios válidos
   - Incluir el individuo baseline (configuración actual) como punto de partida

2. PARA cada generación (80 en total):
   
   a. EVALUACIÓN
      - Calcular fitness de cada individuo sobre los 148 mensajes
   
   b. ELITISMO
      - Copiar los 5 mejores directamente a la nueva población
   
   c. REPRODUCCIÓN (hasta completar 60 individuos):
      - Seleccionar padre1 por torneo
      - Seleccionar padre2 por torneo
      - Cruzar → hijo1, hijo2
      - Mutar hijo1 y hijo2
      - Si son válidos, agregarlos a la nueva población
   
   d. Reemplazar población con la nueva
   
   e. Registrar el mejor fitness de la generación

3. RESULTADO
   - Devolver el mejor individuo encontrado en todas las generaciones
   - Imprimir los parámetros listos para copiar a scoring.py
```


## 8. Convergencia del algoritmo

El siguiente gráfico muestra cómo evoluciona el fitness a lo largo de las generaciones. Es típico ver una mejora rápida al principio y una estabilización progresiva cuando el algoritmo converge.


In [ ]:
# Simulación ilustrativa de convergencia
# (los valores reales se obtienen corriendo optimizar_membresias.py)

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

generaciones = np.arange(1, 81)
baseline     = 0.439

# Curva de convergencia típica
mejora   = 1 - np.exp(-generaciones / 15)
fitness  = baseline + (0.72 - baseline) * mejora
fitness += np.random.normal(0, 0.008, len(generaciones)) * (1 - mejora)
fitness  = np.maximum.accumulate(fitness)  # monotónicamente no decreciente

fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(generaciones, fitness, color='#3498db', linewidth=2, label='Mejor fitness')
ax.axhline(baseline, color='#e74c3c', linestyle='--', linewidth=1.5, label=f'Baseline ({baseline:.3f})')
ax.fill_between(generaciones, baseline, fitness, alpha=0.15, color='#3498db')

ax.set_title('Convergencia del algoritmo genético (ilustrativo)', fontsize=13)
ax.set_xlabel('Generación')
ax.set_ylabel('Accuracy (fitness)')
ax.set_xlim(1, 80)
ax.set_ylim(0.35, 0.80)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Nota: este gráfico es ilustrativo.")
print("Los valores reales se obtienen del output de optimizar_membresias.py")


## 9. ¿Por qué un algoritmo genético y no otra técnica?

### Alternativas consideradas

**Búsqueda exhaustiva (grid search)**  
Probar todas las combinaciones posibles. Con 16 parámetros continuos, es computacionalmente imposible.

**Gradiente descendente**  
Requiere que la función objetivo sea diferenciable. La accuracy sobre un dataset discreto no lo es.

**Optimización bayesiana**  
Más eficiente que el AG pero más compleja de implementar y explicar. Requiere librerías adicionales.

**Ajuste manual**  
Lo que se intentó primero. Funciona hasta cierto punto pero no converge cuando los efectos son interdependientes.

### Por qué el AG es adecuado aquí

- **No requiere derivadas**: la función de fitness puede ser cualquier cosa que devuelva un número
- **Robusto a mínimos locales**: la población mantiene diversidad y puede escapar de óptimos locales
- **Paralelizable**: cada individuo se evalúa independientemente
- **Interpretable**: el proceso es conceptualmente simple y explicable
- **Conección académica directa**: es un tema explícito del cronograma de la materia

### Conexión con el cronograma de la materia

Este proyecto combina **dos temas** del cronograma:

```
Lógica Difusa    →  sistema de detección de toxicidad
                         ↑
Algoritmos Genéticos  →  optimización de parámetros
```

La combinación tiene respaldo en la literatura: el enfoque de optimizar sistemas difusos con AGs se conoce como *Genetic Fuzzy Systems* y es un área activa de investigación desde los años 90.


## 10. Parámetros del algoritmo y justificación

| Parámetro | Valor | Justificación |
|---|---|---|
| `POBLACION_SIZE` | 60 | Balance entre diversidad y costo computacional |
| `GENERACIONES` | 80 | Suficiente para converger sin overfitting al dataset |
| `TASA_MUTACION` | 0.15 | 15% por gen: exploración sin destruir buenos individuos |
| `TASA_CRUCE` | 0.70 | Alta tasa de recombinación para explotar buenas soluciones |
| `ELITE_SIZE` | 5 | 8% de la población: conserva los mejores sin estancar |
| `TORNEO_SIZE` | 4 | Presión selectiva moderada |

### Consideración importante: overfitting

Con solo 148 mensajes etiquetados, existe riesgo de que el algoritmo encuentre parámetros que maximizan la accuracy sobre ese dataset específico pero no generalizan bien a mensajes nuevos.

Para mitigar esto:
- Las **reglas difusas se mantienen fijas** (representan conocimiento del dominio, no se optimizan)
- Solo se optimizan los **umbrales de la variable de salida**, que tienen 16 grados de libertad
- Se mantiene el **criterio humano** como validación final antes de desplegar en el servidor real


## 11. Cómo usar el optimizador

### Prerequisitos

```bash
pip install scikit-fuzzy numpy
```

### Ejecución

```bash
# Con el dataset por defecto (logs/testing.csv)
python optimizar_membresias.py

# Con un dataset específico
python optimizar_membresias.py ruta/al/testing.csv
```

### Output esperado

```
Cargando dataset: logs/testing.csv
  148 mensajes cargados

Iniciando algoritmo genético
  Población: 60 | Generaciones: 80
  Dataset: 148 mensajes

Evaluando configuración actual como baseline...
  Accuracy baseline: 0.4392 (65/148)

  Gen   1/80 | Mejor: 0.5068 (75/148) | Promedio: 0.4319
  Gen   5/80 | Mejor: 0.5473 (81/148) | Promedio: 0.4821
  ...
  Gen  80/80 | Mejor: 0.6824 (101/148) | Promedio: 0.6102

=======================================================
RESULTADO FINAL
=======================================================
Accuracy baseline:   0.4392
Accuracy optimizada: 0.6824
Mejora: +24.3 puntos porcentuales

Parámetros óptimos encontrados:
Copialos en el CONFIG de scoring.py

'toxicidad': {
    'baja':    {'tipo': 'trapezoidal', 'params': (0.000, 0.000, 0.142, 0.271)},
    'media':   {'tipo': 'trapezoidal', 'params': (0.183, 0.294, 0.418, 0.537)},
    'alta':    {'tipo': 'trapezoidal', 'params': (0.401, 0.532, 0.671, 0.779)},
    'extrema': {'tipo': 'trapezoidal', 'params': (0.648, 0.742, 1.000, 1.000)},
},
```

### Integración con scoring.py

Copiar el bloque `'toxicidad': { ... }` generado al CONFIG de `src/scoring.py` y reemplazar los valores anteriores.


## 12. Limitaciones y trabajo futuro

### Limitaciones actuales

**Dataset pequeño**: 148 mensajes etiquetados manualmente. Con más datos, el optimizador encontraría parámetros más robustos y generalizables.

**Solo optimiza la salida**: las membresías de las variables de entrada (lista_negra, CONICET, detoxify, historial) se mantienen fijas por diseño. Una extensión natural sería optimizar también esas membresías, aunque aumentaría el espacio de búsqueda a ~80 parámetros.

**Las reglas no se optimizan**: la estructura de reglas representa conocimiento del dominio definido manualmente. Técnicas como *Pittsburgh approach* o *Michigan approach* optimizarían también las reglas, pero a costa de interpretabilidad.

### Extensiones posibles

- **Optimizar también las membresías de entrada**: aumentaría el espacio de búsqueda pero podría mejorar significativamente los resultados
- **Algoritmo multi-objetivo**: optimizar simultáneamente accuracy y recall por categoría, especialmente para la categoría "extrema" que tiene mayor costo de falso negativo
- **Reentrenamiento continuo**: a medida que el bot en producción acumula más mensajes etiquetados, re-correr el optimizador periódicamente para adaptar el sistema al lenguaje emergente del servidor


## Referencias

- Pérez, J. M., Miguel, P., & Cotik, V. (2024). *Exploring Large Language Models for Hate Speech Detection in Rioplatense Spanish*. Instituto de Ciencias de la Computación, UBA/CONICET.

- Cordón, O., Herrera, F., Hoffmann, F., & Magdalena, L. (2001). *Genetic Fuzzy Systems: Evolutionary Tuning and Learning of Fuzzy Knowledge Bases*. World Scientific.

- Holland, J. H. (1975). *Adaptation in Natural and Artificial Systems*. University of Michigan Press.

- Zadeh, L. A. (1965). Fuzzy sets. *Information and Control*, 8(3), 338–353.

---

*Notebook generado para el proyecto Cosme Botito — Introducción a la IA, UTN*
